# Minimal working example using BERT (`distBERT` model)

In [13]:
!pip install pandas torch transformers faiss-cpu


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: C:\Users\monaa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Setup

In [14]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss

## Pseudo data creation

In [15]:
items = pd.read_csv("items.csv")
events = pd.read_csv("events.csv")
events["ts"] = pd.to_datetime(events["ts"])

In [16]:
events.head(5)


,user_id,item_id,ts,event_type,session_id,device,dwell_seconds,price_at_event,category_at_event
0,u1,i2,2026-01-01 07:47:00,click,s_u1_20260101_1,tablet,56,129.00,electronics
1,u1,i13,2026-01-04 08:01:00,view,s_u1_20260104_2,tablet,7,149.99,electronics
2,u1,i2,2026-01-07 14:28:00,click,s_u1_20260107_3,mobile,72,129.00,electronics
3,u1,i13,2026-01-07 19:06:00,view,s_u1_20260107_1,tablet,67,149.99,electronics
4,u1,i1,2026-01-10 08:42:00,add_to_cart,s_u1_20260110_1,desktop,46,199.99,electronics


## BERT encoder (What makes BERT work?)

In [17]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-uncased"  # for illustration, 66M model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_special_tokens({'pad_token': '[PAD]'})
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
encoder.resize_token_embeddings(len(tokenizer))


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5364.79it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding(30522, 768, padding_idx=0)

**Evaluate the BERT encoder**

In [18]:
# ----------------------------
# 2) BERT encode helper
# ----------------------------

# turn off gradient calculation (no back propagation in using BERT)
@torch.no_grad()
def bert_embed(texts, max_len=256):
    batch = tokenizer(
        texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    out = encoder(**batch)
    cls = out.last_hidden_state[:, 0]          # first column [CLS]-like token for classification
    emb = F.normalize(cls, dim=-1)             # normalization
    return emb.cpu().numpy().astype("float32") # size (B, 768)


## Embedding items into vectors (for comparisons)

In [19]:
# ----------------------------
# 3) Offline job: item embeddings
# ----------------------------

items["text"] = (items["title"] + ". " + items["description"] + ". " + "Brand: " + items["brand"] + ". " + "Tags: " + items["tags"].str.replace(",", ", "))
item_vecs = bert_embed(items["text"].tolist())
item_id_list = items["item_id"].tolist()

# Build ANN index (inner product works with normalized vectors)
index = faiss.IndexFlatIP(item_vecs.shape[1])
index.add(item_vecs)


## What user-specific data are there?

In [20]:
# ----------------------------
# 4) Feature builder: user text from last N clicks
# ----------------------------
def build_user_text(user_id, events, items, N=3):
    meaningful = events[
        (events["user_id"] == user_id) &
        (events["event_type"].isin(["click", "add_to_cart", "purchase"]))
    ]
    hist = (meaningful
            .sort_values("ts")
            .tail(N)["item_id"]
            .tolist())
    if not hist:
        return "no history", set()
    valid_hist = [iid for iid in hist if iid in items["item_id"].values]
    if not valid_hist:
        return "no history", set()
    text = items.set_index("item_id").loc[valid_hist, "text"].tolist()
    return " ".join(text), set(hist)

## How to recommend "similar" item with BERT?

In [21]:
# ----------------------------
# 5) "What to recommend" function
# ----------------------------
def recommend(user_id, k=3):
    user_text, seen = build_user_text(user_id, events, items, N=3)
    u = bert_embed([user_text])  # (1, 768)
    scores, idx = index.search(u, k + len(seen))  # keep track of what was seen by the user
    recs = []
    for j in idx[0]:
        iid = item_id_list[j]
        if iid not in seen:
            recs.append(iid)
        if len(recs) == k:
            break
    return recs


In [22]:

for u in ["u1","u2","u3"]:
    print(u, "->", recommend(u, k=3))


u1 -> ['i14', 'i5', 'i6']
u2 -> ['i18', 'i5', 'i13']
u3 -> ['i3', 'i5', 'i13']
